# Exercises: Inference with PyMC

## 1. Using MCMC for inference on Binary Networks

Given the factorised distribution

$P(C,R,D,A,S,F,T) = P(R)P(C)P(D)P(A\vert C)P(F\vert A)P(T\vert F,D) P(S\vert F,T)$

and the conditional and prior probability tables given in the code cell below (each array is indexed in the same order as its name, so that `PA_C[a][c]` is $P(A=a\vert C=c)$ and `PT_FD[t][f][d]` is $P(T=t\vert F=f,D=d)$), compute the following quantities using `pymc`. Verify your answers against the exact solutions and the sampling solutions computed in 3.7 (Questions 1 and 3).

In [1]:
import numpy as np
PR = np.array([0.7,0.3])
PC = np.array([0.9,0.1])
PD = np.array([0.5,0.5])
PA_C = np.array([[1.0, 0.99],[0.0,0.01]])
PF_A = np.array([[0.999, 0.7],[0.001,0.3]])
PT_FD = np.array([[[0.99,0.40],[0.30,0.01]],[[0.01,0.60],[0.70,0.99]]])
PS_TF = np.array([[[0.999,0.01],[0.70,0.01]],[[0.001,0.99],[0.30,0.99]]])


1. $P(S)$

2. $P(S,C)$

3. $P(A\vert S)$

4. $P(A\vert T)$

## 2. A Multi-valued Problem

Consider the following problem:

You are very proud of your garden and like to keep it well watered, but you don't like to be wasteful. You have a garden sprinkler that you use when it is necessary.


* You are very unlikely to turn on the sprinkler when it is raining, somewhat likely when it is cloudy and very likely when it is sunny.
* Where you live, it is quite likely to be sunny, somewhat likely to be cloudy, and unlikely to be raining.
* Whether the grass is wet depends both on the weather and on whether you have used the sprinkler.
* The sprinkler has 3 settings, `off`, `low`, `high`

We can model this using the following network, with random variables $S$ (sprinkler), $W$ (weather) and $G$ (Grass):


$P(S,G,W) = P(G\vert S,W)P(S\vert W)P(W)$

The probabilities can be estimated as:

| $P(W=0)$ | $P(W=1)$ | $P(W=2)$ |
|:---------|:---------|:---------|
| $0.5$    | $0.3$    | $0.2$    |

with $W=0$: sunny; $W=1$: cloudy; $W=2$: raining

|       | $P(S=0\vert W)$ | $P(S=1\vert W)$ | $P(S=2\vert W)$ |
|:------|:---------|:---------|:---------|
| $W=0$ | $0.1$    | $0.3$    | $0.6$    |
| $W=1$ | $0.7$    | $0.2$    | $0.1$    |
| $W=2$ | $0.9$    | $0.1$    | $0.0$    |

with $S=0$: off; $S=1$: low; $S=2$: high

|            | $P(G=0\vert W,S)$ | $P(G=1\vert W,S)$ | $P(G=2\vert W,S)$ |
|:-----------|:------------------|:------------------|:------------------|
| $W=0, S=0$ | $0.9$             | $0.1$            | $0.0$             |
| $W=0, S=1$ | $0.5$             | $0.4$            | $0.1$             |
| $W=0, S=2$ | $0.2$             | $0.5$            | $0.3$             |
| $W=1, S=0$ | $0.8$             | $0.2$            | $0.0$             |
| $W=1, S=1$ | $0.4$             | $0.5$            | $0.1$             |
| $W=1, S=2$ | $0.2$             | $0.5$            | $0.3$             |
| $W=2, S=0$ | $0.1$             | $0.3$            | $0.6$             |
| $W=2, S=1$ | $0.05$            | $0.25$           | $0.7$             |
| $W=2, S=2$ | $0.02$            | $0.18$           | $0.8$             |

with $G=0$: dry; $G=1$: damp; $G=2$: wet


The code cell below gives the tables as `numpy` arrays. In keeping with our notation each array is indexed in the same order as its name, so `PS_W[s][w]` is $P(S=s\vert W=w)$ and `PG_SW[g][s][w]` is $P(G=g\vert S=s,W=w)$. (The tables above list one row per value of the conditioning variables, so the arrays are transposed relative to the tables.)

Compute the (marginal) distribution $P(G)$ using `pymc`, and check your answer by calculating it exactly, as in 3.6.

In [2]:
import numpy as np

grass_states = ["dry", "damp", "wet"]

# P(W): sunny, cloudy, rainy
PW = np.array([0.5, 0.3, 0.2])

# P(S|W): the table has one row per W and one column per S, so transpose to get PS_W[s][w]
PS_W = np.array([[0.1, 0.3, 0.6],     # W = sunny
                 [0.7, 0.2, 0.1],     # W = cloudy
                 [0.9, 0.1, 0.0]]).T  # W = rainy

# P(G|S,W): rows listed by W then S, columns G, i.e. an array indexed [w][s][g].
# Reorder the axes to PG_SW[g][s][w].
PG_WS = np.array([
    [[0.9, 0.1, 0.0], [0.5, 0.4, 0.1], [0.2, 0.5, 0.3]],       # W = sunny:  S = off, low, high
    [[0.8, 0.2, 0.0], [0.4, 0.5, 0.1], [0.2, 0.5, 0.3]],       # W = cloudy
    [[0.1, 0.3, 0.6], [0.05, 0.25, 0.7], [0.02, 0.18, 0.8]],   # W = rainy
])
PG_SW = np.transpose(PG_WS, (2, 1, 0))
